In [ ]:
import sys
sys.path.append("../../")

from evaluation.eval import product_load_tests, evaluate_retrieval
from evaluation.reporting import export_reranking_report_to_markdown
from retrieval.product_retriever import search_product_hybrid
from retrieval.reranker import rerank
import pandas as pd

In [7]:
tests = product_load_tests()

In [8]:
def retrieving(test, top_k):
    context = search_product_hybrid(test.question, top_k=top_k)
    nodes = getattr(context, "nodes", context)
    reranked_nodes = rerank(test.question, nodes, top_k)
    return nodes, reranked_nodes

In [9]:
rows = []
for index, test in enumerate(tests, 1):
    nodes, reranked_nodes = retrieving(test, 30)
    base_eval_result = evaluate_retrieval(test, nodes, 10)
    rerunked_eval_result = evaluate_retrieval(test, reranked_nodes, 10)
    rows.append({
        'question': base_eval_result['question'],
        'relevant_docs': base_eval_result['relevant_docs'],
        'retrieved_docs': base_eval_result['retrieved_docs'],
        'RERANKED_retrieved_docs': rerunked_eval_result['retrieved_docs'],
        'precision': base_eval_result['precision'],
        'RERANKED_precision': rerunked_eval_result['precision'],
        'recall': base_eval_result['recall'],
        'RERANKED_recall': rerunked_eval_result['recall'],
        'mrr': base_eval_result['mrr'],
        'RERANKED_mrr': rerunked_eval_result['mrr'],
        'ndcg': base_eval_result['ndcg'],
        'RERANKED_ndcg': rerunked_eval_result['ndcg'],
    })


report_df = pd.DataFrame(rows)
report_df

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14558.78it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15834.40it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14242.00it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Ke

,question,relevant_docs,retrieved_docs,RERANKED_retrieved_docs,precision,RERANKED_precision,recall,RERANKED_recall,mrr,RERANKED_mrr,ndcg,RERANKED_ndcg
0,Show me Regular trousers within a budget of 2000.,"[17817750, 18770002, 18769968, 18769944, 18769...","[16279046, 14220194, 13523706, 13647622, 13769...","[16279046, 19011990, 18867438, 18770032, 18769...",0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
1,"Can you find something similar to ""Athena Wome...",[11166548],"[11166548, 16752820, 16154596, 16565622, 10856...","[11166548, 10758214, 11634534, 11634536, 12086...",0.1,0.1,1.0,1.0,1.000000,1.000000,1.0,1.0
2,"Show me products like ""InWeave Women Red Print...",[18921114],"[18921114, 17168254, 15322490, 15447990, 17168...","[18921114, 17168254, 15322490, 17168250, 15542...",0.1,0.1,1.0,1.0,1.000000,1.000000,1.0,1.0
3,I need product with a Woven design pattern in ...,"[12824928, 18262088]","[10451734, 18122478, 19218994, 17663018, 16719...","[18141434, 16719554, 11117288, 19218994, 18122...",0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
4,I'm looking for products from MANGO for everyd...,"[15315768, 15265896, 15265898, 16892568, 15977...","[14006764, 16124612, 17607938, 18321088, 18877...","[17607938, 16124404, 18319598, 16124612, 15274...",0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
95,Do you have Grey Regular with a Solid pattern?,"[8802067, 2075014, 14076362, 14424638, 1851833...","[16311568, 16893050, 16172382, 18489222, 17255...","[14658100, 15855506, 2196419, 18290002, 171555...",0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
96,I need Blue Shirt style by Stylo Bug.,"[18473086, 14673176]","[18473086, 14673176, 17613206, 18839616, 18841...","[18473086, 14673176, 17613206, 18839830, 18841...",0.2,0.2,1.0,1.0,1.000000,1.000000,1.0,1.0
97,Show me products that are Mustard and have Eth...,[17663018],"[18509068, 18135092, 17663018, 14402632, 16945...","[18509068, 18135092, 17663018, 10513948, 18226...",0.1,0.1,1.0,1.0,0.333333,0.333333,0.5,0.5
98,Do you have any Green products for an ethnic l...,"[18085512, 16608648, 17914962, 15160688, 13837...","[16931644, 17589238, 13915012, 18069514, 12867...","[19392350, 17172058, 12024994, 16931644, 14358...",0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0


In [10]:
def build_reranking_markdown_report(report_df, output_path="../../reports/product_reranking_evaluation.md"):
    return export_reranking_report_to_markdown(
        report_df,
        output_path,
        notebook_path="notebooks/product/07_product_reranking.ipynb",
    )

In [11]:
report_path, summary_df = build_reranking_markdown_report(report_df)
print(f"Markdown report saved to: {report_path}")
summary_df

Markdown report saved to: ../../reports/product_reranking_evaluation.md


,questions,precision,RERANKED_precision,recall,RERANKED_recall,mrr,RERANKED_mrr,ndcg,RERANKED_ndcg
0,100,0.166,0.187,0.646,0.675,0.605,0.683,0.573,0.646
